In [ ]:
!pip install gymnasium[atari] ale-py opencv-python wandb --quiet

In [ ]:
import os
import random
from collections import deque

import ale_py
import cv2
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import wandb
from torch.nn import MSELoss

gym.register_envs(ale_py)

PONG_BEST_REWARD = -21

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import argparse

args = argparse.Namespace(
    save_dir        = "./results/vanilla-pong",
    wandb_run_name  = "vanilla-pong",
    batch_size      = 32,
    memory_size     = 100_000,
    lr              = 0.0001,
    discount_factor = 0.99,
    epsilon_start   = 1.0,
    epsilon_decay   = 0.999995,
    epsilon_min     = 0.05,
    target_update_frequency = 1000,
    replay_start_size       = 10_000,
    max_episode_steps       = 27_000,
    train_per_step          = 1,
    episodes                = 5000,
)

print("Hyperparameters set.")

In [ ]:
wandb.login()

In [ ]:
class AtariPreprocessor:
    def __init__(self, frame_stack=4):
        self.frame_stack = frame_stack
        self.frames = deque(maxlen=frame_stack)

    def preprocess(self, obs):
        if len(obs.shape) == 3 and obs.shape[2] == 3:
            gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        else:
            gray = obs
        resized = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
        return resized

    def reset(self, obs):
        frame = self.preprocess(obs)
        self.frames = deque([frame for _ in range(self.frame_stack)], maxlen=self.frame_stack)
        return np.stack(self.frames, axis=0)

    def step(self, obs):
        frame = self.preprocess(obs)
        self.frames.append(frame.copy())
        return np.stack(self.frames, axis=0)

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def append(self, transition):
        self.buffer.append(transition)

    def sample(self, batch_size):
        experiences = random.sample(self.buffer, k=batch_size)
        states, actions, rewards, next_states, dones = zip(*experiences)
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

In [ ]:
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)


class DQN(nn.Module):
    def __init__(self, input_channels, num_actions):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, num_actions),
        )

    def forward(self, x):
        return self.network(x / 255.0)

In [ ]:
class DQNAgent:
    def __init__(self, env_name="ALE/Pong-v5", args=None):
        self.env      = gym.make(env_name, render_mode="rgb_array")
        self.test_env = gym.make(env_name, render_mode="rgb_array")
        self.num_actions = self.env.action_space.n

        self.env.observation_space.seed(seed=seed)
        self.env.action_space.seed(seed=seed)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("Using device:", self.device)

        self.preprocessor = AtariPreprocessor()
        self.memory   = ReplayBuffer(args.memory_size)
        self.q_net    = DQN(self.preprocessor.frame_stack, self.num_actions).to(self.device)
        self.q_net.apply(init_weights)
        self.target_net = DQN(self.preprocessor.frame_stack, self.num_actions).to(self.device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=args.lr)
        self.criterion = MSELoss()

        self.batch_size      = args.batch_size
        self.gamma           = args.discount_factor
        self.epsilon         = args.epsilon_start
        self.epsilon_decay   = args.epsilon_decay
        self.epsilon_min     = args.epsilon_min
        self.episodes        = args.episodes

        self.env_count   = 0
        self.train_count = 0
        self.best_reward = PONG_BEST_REWARD

        self.max_episode_steps      = args.max_episode_steps
        self.replay_start_size      = args.replay_start_size
        self.min_buffer_to_train    = max(self.replay_start_size, self.batch_size)
        self.target_update_frequency = args.target_update_frequency
        self.train_per_step         = args.train_per_step
        self.save_dir               = args.save_dir
        os.makedirs(self.save_dir, exist_ok=True)

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.num_actions - 1)
        state_t = torch.from_numpy(np.array(state)).float().unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.q_net(state_t)
        return q_values.argmax().item()

    def run(self):
        for ep in range(self.episodes):
            obs, _   = self.env.reset(seed=seed + ep)
            state    = self.preprocessor.reset(obs)
            done     = False
            total_reward = 0
            step_count   = 0

            while not done and step_count < self.max_episode_steps:
                action = self.select_action(state)
                next_obs, reward, terminated, truncated, _ = self.env.step(action)
                done = terminated or truncated

                next_state = self.preprocessor.step(next_obs)
                self.memory.append((state, action, reward, next_state, done))

                for _ in range(self.train_per_step):
                    self.train()

                state         = next_state
                total_reward += reward
                self.env_count += 1
                step_count   += 1

                if self.env_count % 10_000 == 0:
                    print(f"[Collect] Ep:{ep} SC:{self.env_count} UC:{self.train_count} Eps:{self.epsilon:.4f}")
                    wandb.log({
                        "Env Step Count": self.env_count,
                        "Update Count":   self.train_count,
                        "Epsilon":        self.epsilon,
                    })

            print(f"[Episode {ep}] Reward:{total_reward:.1f} SC:{self.env_count} Eps:{self.epsilon:.4f}")
            wandb.log({
                "Episode":        ep,
                "Total Reward":   total_reward,
                "Env Step Count": self.env_count,
                "Epsilon":        self.epsilon,
            })

            if ep % 100 == 0:
                path = os.path.join(self.save_dir, f"model_ep{ep}.pt")
                torch.save(self.q_net.state_dict(), path)
                print(f"  → Checkpoint saved: {path}")

            if ep % 20 == 0:
                eval_reward = self.evaluate()
                print(f"  [Eval] Ep:{ep} EvalReward:{eval_reward:.2f}")
                wandb.log({
                    "Eval Reward":     eval_reward,
                    "Env Step Count":  self.env_count,
                })
                if eval_reward > self.best_reward:
                    self.best_reward = eval_reward
                    path = os.path.join(self.save_dir, "best_model.pt")
                    torch.save(self.q_net.state_dict(), path)
                    print(f"  → New best model saved ({eval_reward:.1f})")

    def evaluate(self):
        obs, _  = self.test_env.reset()
        state   = self.preprocessor.reset(obs)
        done    = False
        total   = 0
        while not done:
            s_t = torch.from_numpy(np.array(state)).float().unsqueeze(0).to(self.device)
            with torch.no_grad():
                action = self.q_net(s_t).argmax().item()
            next_obs, reward, terminated, truncated, _ = self.test_env.step(action)
            done  = terminated or truncated
            total += reward
            state  = self.preprocessor.step(next_obs)
        return total

    def train(self):
        if len(self.memory) < self.min_buffer_to_train:
            return

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

        self.train_count += 1

        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        states      = torch.from_numpy(np.array(states,      dtype=np.float32)).to(self.device)
        next_states = torch.from_numpy(np.array(next_states, dtype=np.float32)).to(self.device)
        actions     = torch.tensor(actions, dtype=torch.int64).to(self.device)
        rewards     = torch.tensor(rewards, dtype=torch.float32).to(self.device)
        dones       = torch.tensor(dones,   dtype=torch.float32).to(self.device)

        q_values = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            target_values = self.target_net(next_states).max(1)[0]

        y    = rewards + self.gamma * target_values * (1 - dones)
        loss = self.criterion(q_values, y)

        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        self.optimizer.step()

        if self.train_count % self.target_update_frequency == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())

        if self.train_count % 5000 == 0:
            print(f"  [Train #{self.train_count}] Loss:{loss.item():.4f} Q_mean:{q_values.mean().item():.3f}")
            wandb.log({
                "Loss":         loss.item(),
                "Q mean":       q_values.mean().item(),
                "Train Count":  self.train_count,
            })

In [ ]:
wandb.init(
    project="DLP-Lab5-DQN-Pong",
    name=args.wandb_run_name,
    config=vars(args),
    save_code=True,
)

agent = DQNAgent(env_name="ALE/Pong-v5", args=args)
agent.run()

wandb.finish()

In [ ]:
wandb.save('/kaggle/working/results/vanilla-pong/best_model.pt')